In [1]:
import pandas as pd

# Load your new lightweight dataset
df = pd.read_csv('local_dev_data.csv')

print(f"Total rows: {len(df)}")
# Let's look at exactly what columns IBM gave us
print(df.columns.tolist())
df.head()

Total rows: 1000000
['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/08/01 00:17,20,800104D70,20,800104D70,6794.63,US Dollar,6794.63,US Dollar,Reinvestment,0
1,2022/08/01 00:02,3196,800107150,3196,800107150,7739.29,US Dollar,7739.29,US Dollar,Reinvestment,0
2,2022/08/01 00:17,1208,80010E430,1208,80010E430,1880.23,US Dollar,1880.23,US Dollar,Reinvestment,0
3,2022/08/01 00:03,1208,80010E650,20,80010E6F0,73966883.00,US Dollar,73966883.00,US Dollar,Cheque,0
4,2022/08/01 00:02,1208,80010E650,20,80010EA30,45868454.00,US Dollar,45868454.00,US Dollar,Cheque,0


In [2]:
# The IBM dataset usually has a 'Timestamp' column. 
# We MUST sort by it to prevent data leakage (predicting the past using the future).
df = df.sort_values(by='Timestamp')

# Split: 70% for training the model, 30% for our Kafka simulation later
train_size = int(len(df) * 0.7)

df_train = df.iloc[:train_size].copy()
df_stream = df.iloc[train_size:].copy()

print(f"Training data: {len(df_train)} rows")
print(f"Streaming data: {len(df_stream)} rows")

# Save the streaming portion for Stage 4 (Kafka)
df_stream.to_csv('dev_stream.csv', index=False)

Training data: 700000 rows
Streaming data: 300000 rows


In [3]:
import numpy as np

print("Starting Feature Engineering...")

# 1. Convert Timestamp to an actual DateTime object
df_train['Timestamp'] = pd.to_datetime(df_train['Timestamp'])

# Feature A: Cross-Bank Transfers
# Launderers often move money across different banks to complicate tracking.
df_train['Cross_Bank_Transfer'] = (df_train['From Bank'] != df_train['To Bank']).astype(int)

# Feature B: Time of Day
# Anomalous transactions often happen at 3 AM. Let's extract the hour.
df_train['Hour_of_Day'] = df_train['Timestamp'].dt.hour

# Feature C: Currency Exchange
# Moving money between different currencies is a classic layering technique.
df_train['Currency_Mismatch'] = (df_train['Receiving Currency'] != df_train['Payment Currency']).astype(int)

# Feature D: Log Transform the Amount
# Financial data is highly skewed (lots of $10 transactions, a few $50,000,000 ones).
# Log transformation smooths this out so the model isn't blinded by massive numbers.
df_train['Log_Amount_Paid'] = np.log1p(df_train['Amount Paid'])

# Feature E: Categorical Encoding for Payment Format
# Convert formats like 'Cheque' or 'Reinvestment' into a mathematical matrix (One-Hot Encoding)
payment_dummies = pd.get_dummies(df_train['Payment Format'], prefix='Format')
df_train = pd.concat([df_train, payment_dummies], axis=1)

print("Feature Engineering Complete. New columns added:")
print(df_train.columns.tolist())

Starting Feature Engineering...
Feature Engineering Complete. New columns added:
['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering', 'Cross_Bank_Transfer', 'Hour_of_Day', 'Currency_Mismatch', 'Log_Amount_Paid', 'Format_ACH', 'Format_Bitcoin', 'Format_Cash', 'Format_Cheque', 'Format_Credit Card', 'Format_Reinvestment', 'Format_Wire']


In [6]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix
import joblib

print("--- PART 0: Loading Fresh Data ---")
# Reload the clean data directly from the hard drive to wipe out duplicate columns
df = pd.read_csv('local_dev_data.csv')
df = df.sort_values(by='Timestamp')
train_size = int(len(df) * 0.7)
df_train = df.iloc[:train_size].copy()

print("--- PART 1: Feature Engineering ---")
df_train['Timestamp'] = pd.to_datetime(df_train['Timestamp'])

# Create Features
df_train['Cross_Bank_Transfer'] = (df_train['From Bank'] != df_train['To Bank']).astype(int)
df_train['Hour_of_Day'] = df_train['Timestamp'].dt.hour
df_train['Currency_Mismatch'] = (df_train['Receiving Currency'] != df_train['Payment Currency']).astype(int)
df_train['Log_Amount_Paid'] = np.log1p(df_train['Amount Paid'])

# One-Hot Encoding
payment_dummies = pd.get_dummies(df_train['Payment Format'], prefix='Format')
df_train = pd.concat([df_train, payment_dummies], axis=1)

features_for_model = ['Cross_Bank_Transfer', 'Hour_of_Day', 'Currency_Mismatch', 'Log_Amount_Paid']
features_for_model.extend(payment_dummies.columns.tolist())

print("\n--- PART 2: Preparing Supervised Training Data ---")
X_train = df_train[features_for_model]
y_train = df_train['Is Laundering']

negative_cases = len(y_train[y_train == 0])
positive_cases = len(y_train[y_train == 1])
imbalance_ratio = negative_cases / positive_cases

print(f"Normal Transactions: {negative_cases}")
print(f"Laundering Transactions: {positive_cases}")
print(f"Calculated Imbalance Ratio: {imbalance_ratio:.2f}")

print("\n--- PART 3: Training XGBoost ---")
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=imbalance_ratio, 
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
predictions = xgb_model.predict(X_train)

print("\n--- XGBoost Model Evaluation ---")
print("Confusion Matrix:")
print(confusion_matrix(y_train, predictions))

print("\nClassification Report:")
print(classification_report(y_train, predictions))

joblib.dump(xgb_model, 'aml_xgboost.pkl')
joblib.dump(features_for_model, 'model_features.pkl')

print("\nSUCCESS! Supervised Model saved as 'aml_xgboost.pkl'")

--- PART 0: Loading Fresh Data ---
--- PART 1: Feature Engineering ---

--- PART 2: Preparing Supervised Training Data ---
Normal Transactions: 699980
Laundering Transactions: 20
Calculated Imbalance Ratio: 34999.00

--- PART 3: Training XGBoost ---

--- XGBoost Model Evaluation ---
Confusion Matrix:
[[694474   5506]
 [     0     20]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    699980
           1       0.00      1.00      0.01        20

    accuracy                           0.99    700000
   macro avg       0.50      1.00      0.50    700000
weighted avg       1.00      0.99      1.00    700000


SUCCESS! Supervised Model saved as 'aml_xgboost.pkl'
